In [ ]:
 #Install Required Dependencies
%pip install --upgrade pip

# Uninstall conflicting packages
%pip uninstall -y langchain_classic langchain-core langchain-openai langchain-community langchain langchain-chroma chromadb beautifulsoup4 python-dotenv PyPDF2 rank_bm25 weaviate-client ragas wikipedia langchain-weaviate langchain-together langchain-experimental tiktoken langgraph langchain-tavily

# PRE-STEP: Install Required Dependencies
%pip install langchain==1.1.0
%pip install langgraph==1.0.4
%pip install langchain-openai==1.1.0
%pip install langchain-chroma==1.0.0
%pip install chromadb==1.3.5
%pip install python-dotenv==1.1.1
%pip install pydantic==2.12.3

In [ ]:
# Cell 1: Setup and load the trained agent state from Lab 18-1
import os
import json
import pickle
from datetime import datetime
import pandas as pd
import numpy as np
from typing import Dict, List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from dotenv import load_dotenv

# Import the new architecture components
from coala_agent import CoALAAgent
from domain_investment.investment_advisor_agent import InvestmentAdvisorAgent
from domain_investment.investment_advisor_prompts import (
    PROMPT_MEMORY_OPTIMIZATION, GRADIENT_CRITIQUE, GRADIENT_PROPOSAL,
    METAPROMPT_SURFACE, METAPROMPT_DEEP, METAPROMPT_SYNTHESIS
)
from domain_investment.investor_test_scenarios import (
    run_prompt_memory_test,
    run_gradient_test,
    test_agents_with_queries,
    test_response_consistency,
    compare_agent_performance
)


from domain_agent import DomainProcedure

load_dotenv(dotenv_path='env.txt')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [ ]:
# Cell 2: Define function to save checkpoints
#  Create checkpoint of current state
def save_checkpoint(agent: CoALAAgent, checkpoint_name: str):
    """Save agent state for comparison baseline"""
    # Get procedural memory stats
    proc_stats = agent.procedural_memory.get_stats() if hasattr(agent, 'procedural_memory') else {}
    
    checkpoint = {
        'timestamp': datetime.now().isoformat(),
        'episodic_count': len(agent.vector_store.get()["ids"]) if hasattr(agent.vector_store, 'get') else 0,
        'procedural_stats': proc_stats,
        'learned_strategies': list(agent.procedural_memory.global_procedures.keys()) if hasattr(agent, 'procedural_memory') else [],
        'current_performance': proc_stats.get('avg_success_rate', 0)
    }
    
    with open(f"checkpoints/{checkpoint_name}.pkl", "wb") as f:
        pickle.dump(checkpoint, f)
    
    return checkpoint

# Create checkpoint directory
os.makedirs("checkpoints", exist_ok=True)

print("🔄 Loading trained agent from Lab 18-1...")

# Create domain agent
domain_agent = InvestmentAdvisorAgent()

# IMPORTANT: Load the EXISTING agent from Lab 18-1, not create a new one
existing_memory_dir = os.path.join(domain_agent.domain_dir, "domain_memory_store")

if not os.path.exists(existing_memory_dir):
    print("⚠️  No existing memory store found from Lab 18-1!")
    print("   Please run Lab 18-1 first to create the trained agent.")
    raise FileNotFoundError(f"Memory store not found at {existing_memory_dir}")

# Load the existing trained agent (picking up from Lab 18-1)
baseline_agent = CoALAAgent(
    domain_agent=domain_agent,
    model_name="gpt-4.1-mini",  # Match Lab 18-1
    temperature=0,
    persist_directory=existing_memory_dir  # Use EXISTING memory from Lab 18-1
)

print(f"✅ Loaded existing agent from: {existing_memory_dir}")

# Save baseline checkpoint
baseline_checkpoint = save_checkpoint(baseline_agent, "baseline_lab18_2")

print("\n📊 Current State (from Lab 18-1):")
print(f"  Episodic memories: {baseline_checkpoint['episodic_count']}")
print(f"  Learned strategies: {len(baseline_checkpoint['learned_strategies'])}")
print(f"  Current performance: {baseline_checkpoint['current_performance']:.1%}")

In [ ]:
# Cell 3: Show procedural memory breakdown by scope
# Show breakdown by scope
if hasattr(baseline_agent, 'procedural_memory'):
    stats = baseline_agent.procedural_memory.get_stats()
    print(f"\n  Strategy breakdown by scope:")
    for scope, count in stats['by_scope'].items():
        print(f"    {scope.capitalize()}: {count}")

# Load test data for comparison - use conversations NOT seen in Lab 18-1
data_dir = os.path.join(domain_agent.domain_dir, "investment_advisor_data")
test_conversations = []

if os.path.exists(f"{data_dir}/conversations.jsonl"):
    with open(f"{data_dir}/conversations.jsonl", "r") as f:
        for i, line in enumerate(f):
            # Lab 18-1 used conversations 0-100, so use 100-150 for testing
            if i >= 100 and i < 150:  
                test_conversations.append(json.loads(line))
else:
    print("⚠️  No conversation data found. Run Lab 18-1 first to generate data.")

if test_conversations:
    print(f"\n📁 Loaded {len(test_conversations)} NEW test conversations (not seen in Lab 18-1)")
    print(f"  Success rate in test set: {sum(1 for c in test_conversations if c['feedback']['success']) / len(test_conversations):.1%}")
    print(f"  Avg satisfaction: {sum(c['feedback']['satisfaction_score'] for c in test_conversations) / len(test_conversations):.1f}/5.0")

# Verify we have the trained model
if baseline_checkpoint['episodic_count'] == 0:
    print("\n⚠️  WARNING: Agent appears to be untrained (no memories found)")
    print("   Please complete Lab 18-1 first to train the agent.")
else:
    print(f"\n✅ Ready for optimization testing with {len(test_conversations)} new conversations")
    print("   We'll test 3 LangMem algorithms on this trained agent")

In [ ]:
# Cell 4: Implement and test prompt_memory algorithm
class PromptMemoryOptimizer:
    """
    Prompt_memory: Single-pass optimization with minimal overhead.
    Key characteristics:
    - One LLM call for both analysis and synthesis
    - Lower computational cost
    - Faster adaptation cycles
    - Best for simpler patterns and quick iterations
    """
    
    def __init__(self, llm):
        self.llm = llm
        # Use the investment-specific optimization prompt
        self.optimization_prompt = PromptTemplate.from_template(
            PROMPT_MEMORY_OPTIMIZATION
        )
        self.parser = JsonOutputParser()
    
    def optimize(self, conversations: List[Dict], current_stats: Dict) -> Dict:
        """Single-pass optimization"""
        # Format conversations for analysis
        formatted_convs = []
        for conv in conversations[:10]:  # Limit for context window
            formatted_convs.append({
                "query": conv["messages"][0]["content"],
                "response": conv["messages"][1]["content"][:200],
                "success": conv["feedback"]["success"],
                "satisfaction": conv["feedback"]["satisfaction_score"]
            })
        
        # Single LLM call for optimization
        chain = self.optimization_prompt | self.llm | self.parser
        result = chain.invoke({
            "conversations": json.dumps(formatted_convs, indent=2),
            "current_performance": json.dumps(current_stats)
        })
        
        return {
            "algorithm": "prompt_memory",
            "patterns": result.get("patterns_found", []),
            "rules": result.get("procedural_rules", []),
            "summary": result.get("optimization_summary", ""),
            "llm_calls": 1  # Key efficiency metric
        }
    # TESTING:
# Create new domain agent instance for testing
test_domain_agent = InvestmentAdvisorAgent()

# Initialize optimizer
prompt_optimizer = PromptMemoryOptimizer(
    llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
)

# Run the complete test
test_results = run_prompt_memory_test(
    baseline_agent, 
    test_domain_agent, 
    prompt_optimizer, 
    test_conversations
)

if test_results:
    # Display results
    print(f"\n📊 Prompt_memory Results:")
    print(f"  Time taken: {test_results['optimization_time']:.2f} seconds")
    print(f"  LLM calls: {test_results['prompt_result']['llm_calls']}")
    print(f"  Patterns found: {len(test_results['prompt_result']['patterns'])}") 
    print(f"  Rules generated: {len(test_results['prompt_result']['rules'])}")
    
    if test_results['prompt_result']['rules']:
        print(f"\n  Sample rule: {test_results['prompt_result']['rules'][0]['rule']}")
        print(f"  Priority: {test_results['prompt_result']['rules'][0].get('priority', 'N/A')}")
    
    print(f"\n✅ Applied {test_results['applied_rules']} rules to agent")
    
    # Show efficiency results
    print("\n⏱️ Testing efficiency (3 runs)...")
    eff = test_results['efficiency']
    print(f"  Average optimization time: {eff['avg_time']:.2f} seconds")
    print(f"  Efficiency: {eff['efficiency']:.1f} optimizations/second")
    
    # Save checkpoint
    prompt_checkpoint = save_checkpoint(test_results['agent'], "prompt_memory_test")
    print(f"\n💾 Checkpoint saved: {len(prompt_checkpoint['learned_strategies'])} strategies")